### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from pathlib import Path
# Note: Keep an eye out for dedicated packages replacing these loaders soon
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader 
# Updated import line:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

C:\Users\Fida\AppData\Local\Temp\ipykernel_11484\2296670421.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: attetions.pdf
  ✓ Loaded 15 pages

Processing: nlp.pdf
  ✓ Loaded 23 pages

Processing: rag.pdf
  ✓ Loaded 19 pages

Total documents loaded: 57


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attetions.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attetions.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of T

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 57 documents into 314 chunks

Example chunk:
Content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attetions.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attetions.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attetions.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attetions.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of T

### embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\RAG\learn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9363.30it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 942


In [9]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attetions.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attetions.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of T

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 314 texts...


Batches: 100%|██████████| 10/10 [00:05<00:00,  1.67it/s]


Generated embeddings with shape: (314, 384)
Adding 314 documents to vector store...
Successfully added 314 documents to vector store
Total documents in collection: 1256


### Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 124.97it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_64aac716_12',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'producer': 'pdfTeX-1.40.25',
   'keywords': '',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'moddate': '2024-04-10T21:11:43+00:00',
   'subject': '',
   'title': '',
   'doc_index': 12,
   'page_label': '3',
   'creator': 'LaTeX with hyperref',
   'page': 2,
   'source_file': 'attetions.pdf',
   'file_type': 'pdf',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'total_pages': 15,
   'trapped': '/False',
   'content_length': 216,
   'author': '',
   'source': '..\\data\\pdf\\attetions.pdf'},
  'similarity_score': 0.1399548053741455,
  'distance': 0.8600451946258545,
  'rank': 1},
 {'id': 'doc_882415b2_12',
  'content': '3.2 Attention\nAn a

In [14]:
rag_retriever.retrieve("what is natural language processing")


Retrieving documents for query: 'what is natural language processing'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 166.57it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5367172a_53',
  'content': 'Vision, Automatic Speech Recognition, and in particular, NLP.\nThis survey categorizes and addresses the different aspects and\napplications of NLP that have beneﬁted from deep learning. It\ncovers core NLP tasks and applications, and describes how deep\nlearning methods and models advance these areas. We further\nanalyze and compare different approaches and state-of-the-art\nmodels.\nIndex Terms —Natural Language Processing, Deep Learning,\nArtiﬁcial Intelligence\nI. I NTRODUCTION\nN\nATURAL Language Processing (NLP) is a sub-discipline\nof computer science providing a bridge between natural\nlanguages and computers. It helps empower machines to un-\nderstand, process, and analyze human language [1]. NLP’s sig-\nniﬁcance as a tool aiding comprehension of human-generated\ndata is a logical consequence of the context-dependency\nof data. Data becomes more meaningful through a deeper\nunderstanding of its context, which in turn facilitates text\na

### RAG Pipeline- VectorDB To LLM Output Generation

In [15]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [16]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [17]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

class NvidiaLLM:
    def __init__(self, model_name: str = "nvidia/nemotron-3-ultra-550b-a55b:free", api_key: str = None):
        """
        Initialize Nvidia LLM via OpenRouter.
        
        Args:
            model_name: Nvidia model name hosted on OpenRouter.
            api_key: OpenRouter API key (or set openrai_api_key environment variable).
        """
        self.model_name = model_name
        # Match your existing environment variable naming choice
        self.api_key = api_key or os.environ.get("openrai_api_key")
        
        if not self.api_key:
            raise ValueError("Nvidia API key is required. Set openrai_api_key environment variable or pass api_key parameter.")
        
        # FIXED: Corrected ChatOpenAI parameter names and added OpenRouter base URL
        self.llm = ChatOpenAI(
            model=self.model_name,
            api_key=self.api_key,
            openai_api_base="https://openrouter.ai",
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Nvidia LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context.
        """
        # FIXED: Updated to modern LangChain 0.3 template initialization syntax
        prompt_template = PromptTemplate.from_template(
            """You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt text using kwargs matching the brackets in the template
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting.
        """
        simple_prompt = f"Based on this context: {context}\n\nQuestion: {query}\n\nAnswer:"
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"


In [18]:
# Initialize Nvidia LLM (you'll need to set openrai_api_key environment variable)
try:
    nvidia_llm = NvidiaLLM(api_key=os.getenv("openrai_api_key"))
    print("Nvidia LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your openrai_api_key environment variable to use the LLM.")
    nvidia_llm = None

Initialized Nvidia LLM with model: nvidia/nemotron-3-ultra-550b-a55b:free
Nvidia LLM initialized successfully!


In [19]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("what is natural language processing")

Retrieving documents for query: 'what is natural language processing'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 166.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5367172a_53',
  'content': 'Vision, Automatic Speech Recognition, and in particular, NLP.\nThis survey categorizes and addresses the different aspects and\napplications of NLP that have beneﬁted from deep learning. It\ncovers core NLP tasks and applications, and describes how deep\nlearning methods and models advance these areas. We further\nanalyze and compare different approaches and state-of-the-art\nmodels.\nIndex Terms —Natural Language Processing, Deep Learning,\nArtiﬁcial Intelligence\nI. I NTRODUCTION\nN\nATURAL Language Processing (NLP) is a sub-discipline\nof computer science providing a bridge between natural\nlanguages and computers. It helps empower machines to un-\nderstand, process, and analyze human language [1]. NLP’s sig-\nniﬁcance as a tool aiding comprehension of human-generated\ndata is a logical consequence of the context-dependency\nof data. Data becomes more meaningful through a deeper\nunderstanding of its context, which in turn facilitates text\na

### Integration Vectordb Context pipeline With LLM output

In [20]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY") 
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found. Check your .env file.")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=api_key,
    temperature=0.1,
    max_output_tokens=2048,
    max_retries=3,
)

def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join(doc["content"] for doc in results) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""Use the following context to answer the question concisely.
If the answer isn't in the context, say so.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [21]:
# Run pipeline execution
# Ensure rag_retriever object instance is already built and declared above this line
answer = rag_simple("What is attention mechanism?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 199.48it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'type': 'text', 'text': 'An attention function is a mapping of a query and a set of key-value pairs to an output (where all are vectors), with the output being computed as a weighted sum.', 'extras': {'signature': 'EoMICoAIAWkUfRNzcaQG0pIUIW5uHnAMUetxDJrDnawjf55hzgfCLsmKT02fbbFGxbNM/CgDM66t3ywWBbLdPTj/FN6xp3JtMHRQ3/g1zgljtbyIRbyIqkwc058cM14y/X/s2V6NAx+aTE9CRU7lF3cRXL2R0ymMnH8NOgjUqRkGr985zs+NHGIm8GbGamPOYYEnAlNBPDRHEDs/WReXazdrrXZ5FXyMA/ab8b+dmlTupQ/ckh9VdWNtohZGWVmyuwK1VeyihdxngIZRuADNhmg/jwqBux6ny2xt4Hu8Bf48JJgJq/0UEIw8tK8EUIWI8VhpzCVYMojy6ti1mPpS8tsqUG9e9uV8p0Qv6OIaHrJYJ+h91/wBJT3Vuq6cZhmAkPkcKaQgMM7qKj3h3OSC11Va1fa6M5UFN5Jjh6aK5nqSpQgP9G3OMKDkHMp9PoMBxjqAQnmq3tCkfYY5zdGAd9vVLIHdMGIH+0qEiFn1UtfFJ9hCdSZcgIs9soSfyTE4YOz4SnT+/AFI9ke1VPzqRLBENPBrn+Jv6t3Zc39dBOvvDkpMr10HTsXvMpbKxcaf4GqlTEAlySTgEe/OH6HF9W5hwRxSvyP7V1tJAVIlmqkKfFlqGr2Y0J7Y2B+FCk53pfwaZNIvCvqQ7GTPZzhBLEH355swU8/S2odXtnXGHvd4ERJRRU4PSJRaIQ3P4/jAVf3SY4Zx0STvnE86w75+x5GVbr++ZVJmh6MCgeSuFI3fM5JXVdzUdKsPNILdaLB7DVcojmEVdMMTOOF

In [22]:
from langchain_core.messages import HumanMessage


def get_text(response):
    """Return plain text from a LangChain response, whether content is a str or a list of blocks."""
    content = response.content
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") if isinstance(block, dict) else str(block)
        for block in content
        if not isinstance(block, dict) or block.get("type") == "text"
    )


# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.
If the answer isn't in the context, say so.

Context:
{context}

Question: {query}

Answer:"""
    response = llm.invoke([HumanMessage(content=prompt)])

    output = {
        'answer': get_text(response),   # clean string instead of raw content blocks
        'sources': sources,
        'confidence': confidence,
    }
    if return_context:
        output['context'] = context

    return output


# Example usage:
result = rag_advanced(
    "what is Jeopardy Question Generation?",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True,
)
print("Answer:", result['answer'])
print("\n")
print("Sources:", result['sources'])
print("\n")
print("Confidence:", result['confidence'])
print("\n")
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is Jeopardy Question Generation?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 166.69it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Based on the context, Jeopardy question generation is a knowledge-intensive generation task that involves generating precise, factual Jeopardy questions conditioned on their answer entities.


Sources: [{'source': 'rag.pdf', 'page': 4, 'score': 0.2903452515602112, 'preview': 'Jeopardy is an unusual format that consists of trying to guess an entity from a fact about that entity.\nFor example, “The World Cup” is the answer to the question “In 1986 Mexico scored as the ﬁrst\ncountry to host this international sports competition twice.” As Jeopardy questions are precise,\nfactu...'}, {'source': 'rag.pdf', 'page': 4, 'score': 0.2903452515602112, 'preview': 'Jeopardy is an unusual format that consists of trying to guess an entity from a fact about that entity.\nFor example, “The World Cup” is the answer to the question “In 1986 Mexico scored as the ﬁrst\ncountry to host this international sports competition twice.” As Jeopardy questions are precise,\nfactu...'}, {'source': 'rag.pdf',

In [23]:
result = rag_advanced(
    "what is Attention Mechanism?",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True,
)
print("Answer:", result['answer'])
print("\n")
print("Sources:", result['sources'])
print("\n")
print("Confidence:", result['confidence'])
print("\n")
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is Attention Mechanism?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 142.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Based on the context, an attention function maps a query and a set of key-value pairs to an output (where the query, keys, values, and output are all vectors), with the output computed as a weighted sum.


Sources: [{'source': 'attetions.pdf', 'page': 2, 'score': 0.271417498588562, 'preview': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3...'}, {'source': 'attetions.pdf', 'page': 2, 'score': 0.271417498588562, 'preview': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3...'}, {'source': 'attetions.pdf', 'page': 2, 'score': 0.271417498588562, 'preview': '3.2 Attention\nAn attention function can be described as mapping a query and a set of k